In [ ]:
housePath = "/content/housing.csv"
housePricePath = "/content/HousingAndPrice.csv"

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# @title
class NaiveBayesClassifier:
    def fit(self, trainingTable, correctAnswers):
        trainingTable = np.asarray(trainingTable, dtype=np.float64)
        correctAnswers = np.asarray(correctAnswers)

        self.categories = np.unique(correctAnswers)

        numberOfTrainingTypes = trainingTable.shape[1]
        numberOfCategories = len(self.categories)

        self.averageTrainingValues = np.zeros((numberOfCategories, numberOfTrainingTypes), dtype=np.float64)
        self.trainingSpreads = np.zeros((numberOfCategories, numberOfTrainingTypes), dtype=np.float64)
        self.categoryBaseProbabilities = np.zeros(numberOfCategories, dtype=np.float64)

        for categoryIndex, categoryName in enumerate(self.categories):
            trainingForThisCategory = trainingTable[correctAnswers == categoryName]

            self.averageTrainingValues[categoryIndex, :] = trainingForThisCategory.mean(axis=0)
            self.trainingSpreads[categoryIndex, :] = trainingForThisCategory.var(axis=0) + 1e-9

            totalRows = trainingTable.shape[0]
            categoryRowCount = trainingForThisCategory.shape[0]
            self.categoryBaseProbabilities[categoryIndex] = categoryRowCount / float(totalRows)

    def _calculateTrainingProbability(self, categoryIndex, singleItemTraining):
        avg = self.averageTrainingValues[categoryIndex]
        spread = self.trainingSpreads[categoryIndex]

        numerator = np.exp(-((singleItemTraining - avg) ** 2) / (2 * spread))
        denominator = np.sqrt(2 * np.pi * spread)
        return numerator / denominator

    def _predictSingleItem(self, singleItemTraining):
        categoryScores = []

        for categoryIndex in range(len(self.categories)):
            baseScore = np.log(self.categoryBaseProbabilities[categoryIndex])

            trainingProbabilities = self._calculateTrainingProbability(categoryIndex, singleItemTraining)
            combinedTrainingScore = np.sum(np.log(trainingProbabilities))

            totalScore = baseScore + combinedTrainingScore
            categoryScores.append(totalScore)

        winningIndex = np.argmax(categoryScores)
        return self.categories[winningIndex]

    def predict(self, testingTable):
        testingTable = np.asarray(testingTable, dtype=np.float64)
        return np.array([self._predictSingleItem(itemTraining) for itemTraining in testingTable])


In [ ]:
# @title
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def runKaggleDataset(csvFilePath, targetColumnName):
    print(f"--- Processing: {csvFilePath.split("/")[-1].split(".")[0]} ---")

    df = pd.read_csv(csvFilePath)

    for idCol in ['id', 'Id', 'ID', 'Unnamed: 0']:
        if idCol in df.columns:
            df = df.drop(columns=[idCol])

    df = df.fillna(df.mean(numeric_only=True))

    rawFeatures = df.drop(columns=[targetColumnName])
    correctAnswers = df[targetColumnName]

    trainingTable = pd.get_dummies(rawFeatures, drop_first=True)

    trainingData, testingData, answersTrain, answersTest = train_test_split(
        trainingTable, correctAnswers, test_size=0.2, random_state=42
    )

    classifier = NaiveBayesClassifier()
    classifier.fit(trainingData, answersTrain)
    predictions = classifier.predict(testingData)

    accuracy = accuracy_score(answersTest, predictions) * 100
    print(f"Accuracy: {accuracy:.2f}%\n")

In [ ]:
import kagglehub
heartPath = kagglehub.dataset_download("johnsmith88/heart-disease-dataset") + "/heart.csv"
breastCancerPath = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data") + "/data.csv"
diabetesPath = kagglehub.dataset_download("jamaltariqcheema/pima-indians-diabetes-dataset") + "/diabetes.csv"
irisPath = kagglehub.dataset_download("uciml/iris") + "/Iris.csv"
waterPath = kagglehub.dataset_download("adityakadiwal/water-potability") + "/water_potability.csv"

Using Colab cache for faster access to the 'heart-disease-dataset' dataset.
Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.
Using Colab cache for faster access to the 'pima-indians-diabetes-dataset' dataset.
Using Colab cache for faster access to the 'iris' dataset.
Using Colab cache for faster access to the 'water-potability' dataset.


In [ ]:
runKaggleDataset(heartPath, "target")

--- Processing: heart ---
Accuracy: 80.00%



In [ ]:
runKaggleDataset(breastCancerPath, "diagnosis")

--- Processing: data ---
Accuracy: 62.28%



In [ ]:
runKaggleDataset(diabetesPath, "Outcome")

--- Processing: diabetes ---
Accuracy: 76.62%



In [ ]:
runKaggleDataset(irisPath, "Species")

--- Processing: Iris ---
Accuracy: 100.00%



In [ ]:
runKaggleDataset(waterPath, "Potability")

--- Processing: water_potability ---
Accuracy: 63.11%



In [37]:
# Regression 1

In [40]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [43]:
# @title
class LogisticRegression:
    def __init__(self, learning_rate=0.01, iterations=1000):
        self.lr = learning_rate
        self.iterations = iterations
        self.weights = None
        self.bias = 0
        self.cost_history = []

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def cost(self, h, y):
        m = len(y)
        return - (1/m) * np.sum(y*np.log(h) + (1-y)*np.log(1-h))

    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)

        for _ in range(self.iterations):
            z = np.dot(X, self.weights) + self.bias
            h = self.sigmoid(z)

            dw = (1/m) * np.dot(X.T, (h - y))
            db = (1/m) * np.sum(h - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            self.cost_history.append(self.cost(h, y))

    def predict(self, X):
        return (self.sigmoid(np.dot(X, self.weights) + self.bias) >= 0.5).astype(int)

In [ ]:
df = pd.read_csv('BankNote_Authentication.csv')
df.head()

In [ ]:
X = df.drop(columns=['class']).values
y = df['class'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
print(classification_report(y_test, y_pred))
print(accuracy_score(y_test, y_pred))

In [ ]:
# Regression 2

In [ ]:
df = pd.read_csv('diabetes.csv')
df.head()

In [ ]:
X = df.drop(columns=['Outcome']).values
y = df['Outcome'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [56]:
from sklearn.metrics import classification_report, accuracy_score
print(classification_report(y_test, y_pred))
print(accuracy_score(y_test, y_pred))


                  precision    recall  f1-score   support

           0       0.66      0.98      0.79        99
           1       0.67      0.07      0.13        55

    accuracy                           0.66       154
   macro avg       0.66      0.53      0.46       154
weighted avg       0.66      0.66      0.55       154

0.6558441558441559



In [ ]:
# poly reg

In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()

df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

df.head()

In [45]:
from sklearn.preprocessing import StandardScaler
scalar = StandardScaler()

In [46]:
X = df.drop(columns=['target']).values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

In [47]:
class GDRegressor:
    def __init__(self, lr = 0.1, epochs = 10000):
        self.coef_ = None
        self.intercept_ = None
        self.lr = lr
        self.epochs = epochs

    def fit(self, X_train, y_train):
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])

        for i in range(self.epochs):
            y_hat = np.dot(X_train, self.coef_) + self.intercept_
            intercept_der = -2 * np.mean(y_train - y_hat)
            self.intercept_ = self.intercept_ - (self.lr*intercept_der)
            coef_der = -2 * np.dot((y_train - y_hat), X_train)/X_train.shape[0]
            self.coef_ = self.coef_ - (self.lr * coef_der)

    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_

In [48]:
model = GDRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [49]:
from sklearn.metrics import root_mean_squared_error, r2_score
print(root_mean_squared_error(y_test, y_pred))
print(r2_score(y_test, y_pred))

0.7455813830127763
0.5757877060324508
